# Simple Face Embedding Debug

Compare fresh vs stored embeddings for test faces.

In [1]:
import sys
from pathlib import Path
import numpy as np
import json
import cv2

sys.path.insert(0, str(Path.cwd().parent))
from face_cluster import InsightFaceEmbedder

In [4]:
# Configuration
DATA_DIR = Path(r'D:\sim-bench\results\Google_Germany')
TEST_FACES = [569, 577, 573, 553, 550, 545]  # Target + 5 neighbors

crops_dir = DATA_DIR / 'face_crops'
embeddings_path = DATA_DIR / 'embeddings_FRESH_2026-03-17_01-00-27.npy'
metadata_path = DATA_DIR / 'benchmark_2026-03-01_01-10-04.json'

print(f"Testing faces: {TEST_FACES}")
print(f"Data dir: {DATA_DIR}")

Testing faces: [569, 577, 573, 553, 550, 545]
Data dir: D:\sim-bench\results\Google_Germany


In [ ]:
# Load stored embeddings
print("Loading stored embeddings...")
stored_array = np.load(embeddings_path)

with open(metadata_path) as f:
    metadata = json.load(f)

# Handle None face_index values - use array position as fallback
face_ids = [i if meta.get('face_index') is None else meta.get('face_index') 
            for i, meta in enumerate(metadata['face_metadata'])]
stored = {fid: stored_array[i] for i, fid in enumerate(face_ids)}

print(f"Loaded {len(stored)} stored embeddings")
print(f"Face ID range: {min(stored.keys())} to {max(stored.keys())}")
print(f"Sample face IDs: {list(stored.keys())[:10]}")
print(f"Sample stored embedding shape: {stored[TEST_FACES[0]].shape}")

In [ ]:
# Extract fresh embeddings
print("Extracting fresh embeddings...")
embedder = InsightFaceEmbedder(model_name='buffalo_l', ctx_id=-1)

fresh = {}
for face_id in TEST_FACES:
    img_path = crops_dir / f'face_{face_id:04d}_aligned.jpg'
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    embedding = embedder.get_embedding(img_rgb)
    fresh[face_id] = embedding
    print(f"Face {face_id}: norm={np.linalg.norm(embedding):.4f}")

print(f"\nExtracted {len(fresh)} fresh embeddings")

In [ ]:
# Compare distances
print("Comparing distances from face 569:")
print(f"{'Neighbor':<10} {'Fresh':<12} {'Stored':<12} {'Diff':<12} {'Status'}")
print("-"*60)

target = TEST_FACES[0]
for neighbor in TEST_FACES[1:]:
    # Fresh distance (cosine)
    fresh_dist = 1 - np.dot(fresh[target], fresh[neighbor]) / (
        np.linalg.norm(fresh[target]) * np.linalg.norm(fresh[neighbor])
    )
    
    # Stored distance (cosine)
    stored_dist = 1 - np.dot(stored[target], stored[neighbor]) / (
        np.linalg.norm(stored[target]) * np.linalg.norm(stored[neighbor])
    )
    
    diff = abs(fresh_dist - stored_dist)
    status = "✅ Match" if diff < 0.01 else "❌ MISMATCH"
    
    print(f"{neighbor:<10} {fresh_dist:<12.6f} {stored_dist:<12.6f} {diff:<12.6f} {status}")

In [ ]:
# Display face images
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(TEST_FACES), figsize=(15, 3))
for i, face_id in enumerate(TEST_FACES):
    img_path = crops_dir / f'face_{face_id:04d}_aligned.jpg'
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"Face {face_id}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()